In [1]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.optimizers import Adam

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/Colab Notebooks/섬유요소설계_26년1학기

In [ ]:
df = pd.read_csv('Color.csv', header=0, index_col=None)                         # pandas
df = df.replace(" ", np.nan)
df = df.fillna(0.0)
df

In [ ]:
xy_data = np.array(df, dtype=np.float32)
xy_data

In [ ]:
x_data = xy_data[:, :-1]
y_data = xy_data[:, [-1]]

print(x_data.shape, y_data.shape)

In [6]:
from sklearn.model_selection import train_test_split

x_data, x_test, y_data, y_test = train_test_split(x_data, y_data, train_size=0.8, shuffle=True)

In [ ]:
col_no = len(x_data[0])
col_no

In [ ]:
model = Sequential()

model.add(Input(shape=(col_no,)))
model.add(Dense(1, activation = 'linear'))
model.compile(optimizer=Adam(learning_rate=0.1), loss = 'mse')                       # 추가설명 필요

model.summary()

In [ ]:
from IPython.display import clear_output

class ClearOutputCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 50 == 0:
            clear_output()

hist = model.fit(x_data, y_data, epochs=5000, callbacks=[ClearOutputCallback()])

In [ ]:
plt.figure(figsize=(5,3))
plt.plot(hist.history['loss'])
plt.xlabel('epochs')
plt.ylabel('loss(log)')
plt.yscale('log')
plt.grid()
plt.show()                        # A조 여기까지 실행

In [ ]:
predict_data = model.predict(x_data)

plt.figure(figsize=(5,4))
plt.plot(y_data, predict_data, 'ro')

plt.xlabel('Real Value')
plt.ylabel('Predicted Value')
plt.grid()
plt.show()

In [ ]:
predict_test = model.predict(x_test)                # train_test_split

plt.figure(figsize=(5,4))
plt.plot(y_data, predict_data, 'ro', label='data_set')
plt.plot(y_test, predict_test, 'bo', label='test_set')

plt.xlabel('Real Value')
plt.ylabel('Predicted Value')
plt.legend(loc='best')
plt.grid()
plt.show()

In [ ]:
model.weights

In [ ]:
new_input_data = []
feature_names = df.columns[:-1]

print("각 염료의 사용량을 입력하세요:")
for feature in feature_names:
    while True:
        try:
            value = float(input(f"Enter value for {feature}: "))
            new_input_data.append(value)
            break
        except ValueError:
            print("Invalid input. Please enter a numerical value.")

new_input_array = np.array(new_input_data).reshape(1, -1)
predicted_value = model.predict(new_input_array)
print("\n")
print(f"***** 예상되는 내광성은 다음과 같습니다(최대: 256) => {predicted_value[0][0]:.3f} *****")